## Imports

In [38]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from imblearn.under_sampling import RandomUnderSampler
import pickle

## Utils

In [39]:
def get_data_root():
  '''
    Ritorna il percorso della cartella contenente i dati in base all'ambiente di esecuzione.
  '''
  try:
      import google.colab
      from google.colab import drive

      try:
          drive.mount("/content/drive", force_remount=True)
          return "/content/drive/MyDrive/ColabContent/Data_analytics"
      except Exception:
          print("Drive non montabile")
          return "/content"

  except ImportError:
      return "../../data"

print(get_data_root())

../../data


## Global variables

In [40]:
DATA_ROOT = get_data_root()
DATASET_PATH = f"{DATA_ROOT}/Dataset2526/train.csv"
ONLY_LABEL_ENCODING = False
SEED = 42

# Preprocessing

In [41]:
df = pd.read_csv(DATASET_PATH)

df, df_test = train_test_split(df, test_size=0.2, random_state=SEED)
df_val, df_test = train_test_split(df_test, test_size=0.5, random_state=SEED)

df.to_csv(f"{DATA_ROOT}/train_raw.csv", index=False)
df_val.to_csv(f"{DATA_ROOT}/val_raw.csv", index=False)
df_test.to_csv(f"{DATA_ROOT}/test_raw.csv", index=False)

### Feature selection

In [42]:
# Drop colonne con troppi nan
threshold = 0.25
missing_percent = df.isnull().mean()
cols_to_drop_nan = missing_percent[missing_percent > threshold].index
df_cleaned = df.drop(columns=cols_to_drop_nan)

print(f"Droppate {len(cols_to_drop_nan)} colonne con nan > {threshold*100}%")

Droppate 56 colonne con nan > 25.0%


In [43]:
# Drop colonne poco significative
cols_to_drop_manual = [
    # Testo
    'loan_title',

    # Post-Emissione
    'recoveries_cash', 'collection_recovery_fee', 'total_received_late_fees',
    'loan_status_current_code',
]

df_cleaned = df_cleaned.drop(columns=[c for c in cols_to_drop_manual if c in df_cleaned.columns])

print(f"Droppate {len(cols_to_drop_manual)} colonne poco significative")

Droppate 5 colonne poco significative


### Gestione valori mancanti

In [44]:
num_cols = df_cleaned.select_dtypes(include=[np.number]).columns
cat_cols = df_cleaned.select_dtypes(include=['object']).columns

# Mediana per feature numeriche
for col in num_cols:
  df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].median())

# Moda per feature categoriche
for col in cat_cols:
  if col != 'grade':
    df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].mode()[0])

### Gestione date

In [45]:
# Map per employment_length
emp_len_map = {
    '< 1 year': 0,
    '1 year': 1,
    '2 years': 2,
    '3 years': 3,
    '4 years': 4,
    '5 years': 5,
    '6 years': 6,
    '7 years': 7,
    '8 years': 8,
    '9 years': 9,
    '10+ years': 10
}
df_cleaned['borrower_profile_employment_length'] = df_cleaned['borrower_profile_employment_length'].map(emp_len_map)

# Trasformazione delle date
df_cleaned['loan_issue_date'] = pd.to_datetime(df_cleaned['loan_issue_date'])
df_cleaned['credit_history_earliest_line'] = pd.to_datetime(df_cleaned['credit_history_earliest_line'])

# Calcolo esperienza finanziaria debitore
df_cleaned['months_credit_history'] = (df_cleaned['loan_issue_date'].dt.year - df_cleaned['credit_history_earliest_line'].dt.year) * 12 + \
                              (df_cleaned['loan_issue_date'].dt.month - df_cleaned['credit_history_earliest_line'].dt.month)

# Estrazione mese/anno delle date
date_cols = ['loan_issue_date', 'credit_history_earliest_line', 'last_payment_date', 'last_credit_pull_date']
for col in date_cols:
    if col in df_cleaned.columns:
        date_dt = pd.to_datetime(df_cleaned[col])
        df_cleaned[col + '_year'] = date_dt.dt.year
        df_cleaned[col + '_month'] = date_dt.dt.month

df_cleaned = df_cleaned.drop(columns=date_cols, errors='ignore')

/var/folders/93/wjn3tmxx3694z231t8jw9z280000gn/T/ipykernel_47244/3245697033.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_cleaned['loan_issue_date'] = pd.to_datetime(df_cleaned['loan_issue_date'])
/var/folders/93/wjn3tmxx3694z231t8jw9z280000gn/T/ipykernel_47244/3245697033.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_cleaned['credit_history_earliest_line'] = pd.to_datetime(df_cleaned['credit_history_earliest_line'])
/var/folders/93/wjn3tmxx3694z231t8jw9z280000gn/T/ipykernel_47244/3245697033.py:29: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  date_dt = p

### Gestione outlier

In [46]:
df_out = df_cleaned.copy()
bounds = {}
num_cols = df_out.select_dtypes(include=[np.number]).columns
fico_cols = [c for c in df_out.columns if 'fico' in c.lower()]
perc_cols = [
  'revolving_utilization', 'bankcard_utilization', 'installment_utilization',
  'overall_utilization', 'bankcard_util_gt_75_ratio', 'tradelines_never_delinquent_ratio',
  'loan_contract_interest_rate'
]

for col in num_cols:
  Q1 = df_out[col].quantile(0.25)
  Q3 = df_out[col].quantile(0.75)
  IQR = Q3 - Q1

  # Applichiamo il clipping solo se l'IQR è maggiore di 0
  if IQR > 0:
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df_out[col] = df_out[col].clip(lower=lower_bound, upper=upper_bound)
  # Altrimenti solo valori estremi
  else:
    upper_bound = df_out[col].quantile(0.95)
    df_out[col] = df_out[col].clip(upper=upper_bound)
  
  bounds[col] = (upper_bound, lower_bound)

# Gestione valori FICO >850
for col in fico_cols:
  df_out[col] = df_out[col].clip(upper=850)

# Gestione percentuali >100%
for col in perc_cols:
  if col in df_out.columns:
    # Check se percentuale 0-1 o 0-100
    if df_out[col].max() > 1:
      df_out[col] = df_out[col].clip(upper=100)
    else:
      df_out[col] = df_out[col].clip(upper=1.0)

### Drop colonne con varianza 0

In [47]:
cols_to_drop_const = [col for col in df_out.columns if df_out[col].nunique() <= 1]
df_out = df_out.drop(columns=cols_to_drop_const)

print(f"Droppate {len(cols_to_drop_const)} colonne con varianza 0")

Droppate 7 colonne con varianza 0


### Encoding

In [48]:
# Target
grade_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6}
df_out['grade'] = df_out['grade'].map(grade_map)

numeric_features = df_out.drop(columns=['grade']).select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df_out.drop(columns=['grade']).select_dtypes(include=['object']).columns.tolist()

# Dizionari per salvare gli oggetti
oh_encoders = {}
l_encoders = {}

for col in categorical_features:
    le = LabelEncoder()
    le_data = le.fit_transform(df_out[col].astype(str))
    l_encoders[col] = le
    
    unique_vals = len(le.classes_)

    if unique_vals < 15 and not ONLY_LABEL_ENCODING:
        ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first')
        ohe.fit(df_out[[col]])
        encoded_data = ohe.transform(df_out[[col]])
        column_names = [f"{col}_{cat}" for cat in ohe.categories_[0][1:]]
        
        encoded_df = pd.DataFrame(encoded_data, columns=column_names, index=df_out.index)
        df_out = pd.concat([df_out.drop(col, axis=1), encoded_df], axis=1)
        
        oh_encoders[col] = ohe
        print(f"Colonna '{col}': applicato One-Hot ({unique_vals} cat)")
    else:
        df_out[col] = le_data
        print(f"Colonna '{col}': applicato Label ({unique_vals} cat)")

Colonna 'loan_contract_term_months': applicato One-Hot (2 cat)
Colonna 'borrower_housing_ownership_status': applicato One-Hot (6 cat)
Colonna 'borrower_income_verification_status': applicato One-Hot (3 cat)
Colonna 'loan_payment_plan_flag': applicato One-Hot (2 cat)
Colonna 'loan_purpose_category': applicato One-Hot (14 cat)
Colonna 'borrower_address_zip': applicato Label (874 cat)
Colonna 'borrower_address_state': applicato Label (50 cat)
Colonna 'listing_initial_status': applicato One-Hot (2 cat)
Colonna 'application_type_label': applicato One-Hot (2 cat)
Colonna 'hardship_flag_indicator': applicato One-Hot (2 cat)
Colonna 'disbursement_method_type': applicato One-Hot (2 cat)
Colonna 'debt_settlement_flag_indicator': applicato One-Hot (2 cat)


### Drop colonne con alta correlazione

In [49]:
# Matrice di correlazione 
corr_matrix = df_out.drop(columns=['grade']).corr().abs()
# Selezione triangolo superiore
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop_corr = [column for column in upper.columns if any(upper[column] > 0.95)]
df_out = df_out.drop(columns=to_drop_corr)

print(f"Rimosse {len(to_drop_corr)} colonne per alta correlazione: {to_drop_corr}")

Rimosse 8 colonne per alta correlazione: ['loan_portfolio_total_funded', 'investor_side_funded_amount', 'outstanding_principal_investor_side', 'total_payment_investor_side', 'revolving_tradelines_balance_gt_0', 'satisfactory_accounts', 'total_high_credit_limit', 'credit_history_earliest_line_year']


### Drop colonne poco informative

In [50]:
X_mi = df_out.drop(columns=['grade'])
y_mi = df_out['grade']

mi_scores = mutual_info_classif(X_mi, y_mi, discrete_features='auto', random_state=SEED)

mi_df = pd.DataFrame({'feature': X_mi.columns, 'mutual_info': mi_scores})
mi_df = mi_df.sort_values(by='mutual_info', ascending=False)

threshold_mi = 0.01
low_mi_features = mi_df[mi_df['mutual_info'] < threshold_mi]['feature'].tolist()
df_out = df_out.drop(columns=low_mi_features)
print(f"\nDroppate {len(low_mi_features)} feature con MI < {threshold_mi}")


Droppate 52 feature con MI < 0.01


## Salvataggio asset

In [51]:
final_columns = df_out.drop(columns=['grade']).columns.tolist()
print(f"Colonne finali: {final_columns}")

final_numeric_features = [c for c in final_columns if c in numeric_features]
final_categorical_features = [c for c in final_columns if c in categorical_features]

with open('preprocessing_assets.pkl', 'wb') as f:
    pickle.dump({
        'final_columns': final_columns,
        'numeric_features': final_numeric_features,
        'categorical_features': final_categorical_features,
        'oh_encoders': oh_encoders,
        'l_encoders': l_encoders,
        'clipping_bounds': bounds,
        'medians': df_out.median().to_dict(),
        'modes': df_out.mode().to_dict(),
        'emp_len_map': emp_len_map,
        'grade_map': grade_map
    }, f)

Colonne finali: ['loan_contract_approved_amount', 'loan_contract_interest_rate', 'loan_payment_installments_count', 'borrower_income_annual', 'borrower_dti_ratio', 'fico_score_low_bound', 'fico_score_high_bound', 'credit_inquiries_6m', 'revolving_utilization', 'outstanding_principal_balance', 'total_payment_received', 'total_received_principal', 'total_received_interest', 'last_payment', 'last_fico_score_high_bound', 'last_fico_score_low_bound', 'total_revolving_high_credit_limit', 'accounts_open_past_24m', 'bankcard_open_to_buy', 'bankcard_utilization', 'months_since_oldest_revolving_acct', 'months_since_recent_revolving_acct', 'months_since_recent_trade_line', 'months_since_recent_inquiry', 'tradelines_open_past_12m', 'tradelines_never_delinquent_ratio', 'bankcard_util_gt_75_ratio', 'total_bankcard_credit_limit', 'loan_issue_date_year', 'last_payment_date_year', 'last_credit_pull_date_year', 'last_credit_pull_date_month', 'loan_contract_term_months_ 60 months', 'borrower_income_verif

## Bilanciamento

In [52]:
X = df_out.drop(columns=['grade'])
y = df_out['grade']

rus = RandomUnderSampler(random_state=SEED)
X_res, y_res = rus.fit_resample(X, y)

df_balanced = pd.DataFrame(X_res, columns=X.columns)
df_balanced['grade'] = y_res

print("Distribuzione dopo RandomUnderSampler:")
print(y_res.value_counts())

Distribuzione dopo RandomUnderSampler:
grade
0    4821
1    4821
2    4821
3    4821
4    4821
5    4821
6    4821
Name: count, dtype: int64


## Salvataggio train set preprocessato

In [53]:
if ONLY_LABEL_ENCODING:
    df_balanced.to_csv(f"{DATA_ROOT}/train_processed_only_label.csv", index=False)
else:
    df_balanced.to_csv(f"{DATA_ROOT}/train_processed.csv", index=False)

## Applicazione preprocessing a validation e test set

In [54]:
def preprocess(df_input, assets_path='preprocessing_assets.pkl', use_label_only=False):
    with open(assets_path, 'rb') as f:
        assets = pickle.load(f)
    
    df = df_input.copy()

    # 1. TRASFORMAZIONI INIZIALI
    if 'borrower_profile_employment_length' in df.columns:
        df['borrower_profile_employment_length'] = df['borrower_profile_employment_length'].map(assets['emp_len_map']).fillna(0)

    # Creazione feature temporali 
    date_cols = ['loan_issue_date', 'credit_history_earliest_line', 'last_payment_date', 'last_credit_pull_date']
    for col in date_cols:
        if col in df.columns:
            dt = pd.to_datetime(df[col], errors='coerce')
            df[f"{col}_year"] = dt.dt.year
            df[f"{col}_month"] = dt.dt.month

    if 'loan_issue_date' in df.columns and 'credit_history_earliest_line' in df.columns:
        issue_dt = pd.to_datetime(df['loan_issue_date'], errors='coerce')
        hist_dt = pd.to_datetime(df['credit_history_earliest_line'], errors='coerce')
        df['months_credit_history'] = (issue_dt.dt.year - hist_dt.dt.year) * 12 + (issue_dt.dt.month - hist_dt.dt.month)

    # 2. IMPUTAZIONE 
    for col, value in assets['medians'].items():
        if col in df.columns:
            df[col] = df[col].fillna(value)
            
    for col, values in assets['modes'].items():
        if col in df.columns and col != 'grade':
            # Supporto sia per dizionari di liste che di valori singoli
            val = values[0] if isinstance(values, (list, np.ndarray)) else values
            df[col] = df[col].fillna(val)

    # 3. CLIPPING OUTLIER
    bounds = assets.get('clipping_bounds', {})
    for col, (low, high) in bounds.items():
        if col in df.columns:
            df[col] = df[col].clip(lower=low, upper=high)

    # Clipping percentuali
    perc_cols = ['revolving_utilization', 'bankcard_utilization', 'installment_utilization', 'overall_utilization']
    for col in perc_cols:
        if col in df.columns:
            # Usiamo la mediana del train per decidere la scala
            limit = 100.0 if assets['medians'].get(col, 0) > 1 else 1.0
            df[col] = df[col].clip(upper=limit)
    
    # Clipping FICO
    fico_cols = [c for c in df.columns if 'fico' in c.lower()]
    for col in fico_cols:
        if col in df.columns:
            df[col] = df[col].clip(upper=850)

    # 4. ENCODING

    # One-hot    
    for col, ohe in assets['oh_encoders'].items():
        if col in df.columns:
            if use_label_only:
                if col in assets['l_encoders']:
                    le = assets['l_encoders'][col]
                    df[col] = df[col].astype(str).map(lambda x: le.transform([x])[0] if x in le.classes_ else -1)
                else:
                    pass
            else:
                encoded_data = ohe.transform(df[[col]])
                column_names = [f"{col}_{cat}" for cat in ohe.categories_[0][1:]]
                encoded_df = pd.DataFrame(encoded_data, columns=column_names, index=df.index)
                df = pd.concat([df.drop(col, axis=1), encoded_df], axis=1)
    
    # LabelEncoder
    for col, le in assets['l_encoders'].items():
        if col in df.columns and (not use_label_only or col not in assets['oh_encoders']):
            df[col] = df[col].astype(str).map(lambda x: le.transform([x])[0] if x in le.classes_ else -1)

    # 5. ALLINEAMENTO FINALE 
    final_cols = assets['final_columns']
    for col in final_cols:
        if col not in df.columns:
            df[col] = 0 # Riempie colonne One-Hot mancanti nel batch
            
    # Selezione e pulizia finale NaN residui
    df_final = df[final_cols].copy()
    df_final = df_final.fillna(0) 

    if 'grade' in df.columns:
        df_final['grade'] = df['grade'].map(assets['grade_map'])
    
    return df_final

### Salva val e test

In [55]:
df_val_processed = preprocess(df_val, use_label_only=ONLY_LABEL_ENCODING)
df_test_processed = preprocess(df_test, use_label_only=ONLY_LABEL_ENCODING)
    
if not ONLY_LABEL_ENCODING:
    df_val_processed.to_csv(f"{DATA_ROOT}/val_processed.csv", index=False)
    df_test_processed.to_csv(f"{DATA_ROOT}/test_processed.csv", index=False)
else:
    df_val_processed.to_csv(f"{DATA_ROOT}/val_processed_only_label.csv", index=False)
    df_test_processed.to_csv(f"{DATA_ROOT}/test_processed_only_label.csv", index=False)

/var/folders/93/wjn3tmxx3694z231t8jw9z280000gn/T/ipykernel_47244/1674062270.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(df[col], errors='coerce')
/var/folders/93/wjn3tmxx3694z231t8jw9z280000gn/T/ipykernel_47244/1674062270.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(df[col], errors='coerce')
/var/folders/93/wjn3tmxx3694z231t8jw9z280000gn/T/ipykernel_47244/1674062270.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(df[col], errors='coerce')
/var/folders/93/wjn3tmxx3694z231t8jw9z280000gn/T/ipyker